# Beams in a tokamak

The same coupling as in W7-X, on an axisymmetric equilibrium: freeofcharge's Solov'ev equilibrium (an ASDEX-Upgrade-sized D-shape), written as a G-EQDSK file that TORAX reads directly, and a tangential deuterium beam. freeofcharge labels the flux surfaces of an axisymmetric equilibrium by the poloidal flux, TORAX by the toroidal flux; `NBICoupling` maps the profiles between the two radii with $\Phi = \int q\, d\psi$ (`slowcooker.nbi.toroidal_radius`).

In [ ]:
import logging

logging.disable(logging.INFO)  # TORAX logs every restart at INFO

In [ ]:
import tempfile

import freeofcharge as foc
import matplotlib.pyplot as plt
import numpy as np
import plasma_plots  # noqa: F401

from slowcooker import base_config, merge, profiles, run_coupled
from slowcooker.geometry import eqdsk_geometry
from slowcooker.nbi import NBICoupling, toroidal_radius

equilibrium = foc.Equilibrium.solovev(1.65, 0.5, 1.6, 2.5, q_axis=1.2)
geometry = eqdsk_geometry(equilibrium, tempfile.mkdtemp())
rho_pol = np.linspace(0.0, 1.0, 11)
print("rho_pol:", np.round(rho_pol, 2))
print("rho_tor:", np.round(toroidal_radius(equilibrium, rho_pol), 3))

A 60 keV deuterium beam of 2.5 MW, tangent to $R = 1.3$ m, and the plasma held at a flat density of $4\times10^{19}$ m$^{-3}$ with $\chi = 0.5$ m$^2$/s. The equilibrium carries its own plasma current (the Solov'ev current, from Ampère's law around the boundary), and the rotational transform is held fixed.

In [ ]:
origin, direction = foc.tangential_injection(1.3, 3.5)
beam = foc.NeutralBeam.gaussian(
    origin,
    direction,
    60e3,
    2.5e6,
    mass_number=2,
    width=0.1,
    divergence=0.01,
    markers=2000,
    seed=1,
)
config = merge(
    base_config(geometry, density=4e19, chi=0.5, t_final=0.6, dt=0.05),
    {"sources": {"generic_heat": None}},
)
beams = NBICoupling(equilibrium, beam, rho_edges=np.linspace(0.0, 1.0, 16))
data = profiles(run_coupled(config, [beams], [0.0, 0.2, 0.4, 0.6]))
for start, result in beams.results.items():
    print(
        f"t = {start:.1f} s: absorbed {float(result.absorbed_power) / 1e6:.2f} MW, "
        f"shine-through {100 * float(result.shine_through_fraction):.1f} %, "
        f"driven current {float(result.driven_current) / 1e3:.0f} kA"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
data.p_nbi_i.plasma.plot.profiles(
    x="rho",
    over="t",
    at=[0.1, 0.5],
    ax=axes[0],
    title="beam heating of the ions [W/m$^3$]",
)
data.T_i.plasma.plot.profiles(over="t", ax=axes[1], title="$T_i$ [keV]")